Code-only archive: embedded input/reference bundles were omitted. Run the adjacent build_notebook.py or build_notebooks.py with your local inputs to recreate the executable Kaggle notebook.


# PROTEUS metric research — wave1

Private T4 experiment;no competition submission. All10public people:10LOSO folds and later sessions of2people.

Models:authorShallow40,class loss;Shallow16,class loss;Shallow16,subject/class-balanced loss. AdamW.001,WD.01,dropout.5,causalEAprior5. Inner people-held-out selection,max30epochs,patience6;fresh refit for selected epoch count. Train/val curves saved.

Seed0 all configurations first;seed1 compact models if budget permits. Two workers on separate GPUs;75min scheduling budget with checkpoint preservation. Partial model/seed sets are explicitly labeled.

In [ ]:
import os,sys,pathlib,json,hashlib,zipfile,io,base64,subprocess
os.environ.update(OPENBLAS_NUM_THREADS="1",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1",PYTHONUNBUFFERED="1",MPLCONFIGDIR="/tmp/metric-mpl")
import torch
assert torch.cuda.is_available(),"Enable requested Kaggle GPU"
print("GPU",torch.cuda.get_device_name(0),flush=True)
project=pathlib.Path("/kaggle/working/riemann-project");project.mkdir(exist_ok=True)
sources=list(pathlib.Path("/kaggle/input").rglob("source-integrity.json"));assert len(sources)==1
raw=sources[0].read_bytes()
assert hashlib.sha256(raw).hexdigest()=="3e8cf0b952a1a18ab496a89d6a31275522e8efa003ea6b8ed4b9e6467d2a3c81"
for rel,digest in json.loads(raw).items():
    p=pathlib.PurePosixPath(rel);assert not p.is_absolute() and ".." not in p.parts
    data=(sources[0].parent/rel).read_bytes();assert hashlib.sha256(data).hexdigest()==digest
    dest=project/rel;dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(data)
payload=base64.b64decode(__import__("pathlib").Path("local-bundle.base64").read_text())
assert hashlib.sha256(payload).hexdigest()=="8c785a6e410628ecbf71b71558297ae6c25b4fadbd81db79933cb6cd0262d0b2"
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    manifest=json.loads(z.read("metric-manifest.json"))
    for rel,digest in manifest.items():
        p=pathlib.PurePosixPath(rel);assert not p.is_absolute() and ".." not in p.parts
        data=z.read(rel);assert hashlib.sha256(data).hexdigest()==digest
        dest=project/rel;dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(data)
repro=project/"reproduction";experiment=repro/"metric_research_20261108"
inputs=list(pathlib.Path("/kaggle/input").rglob("input-integrity.json"));assert len(inputs)==1
input_manifest=json.loads(inputs[0].read_text());input_dir=inputs[0].parent
for name,digest in input_manifest["files_sha256"].items():
    h=hashlib.sha256()
    with (input_dir/name).open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):h.update(block)
    assert h.hexdigest()==digest,name
(repro/"data").mkdir(exist_ok=True)
for split in ["train","val","test"]:
    (repro/"data"/f"proteus_{split}.npz").symlink_to(input_dir/f"proteus_{split}.npz")
(repro/"temporal_spd_research/cache").symlink_to(input_dir,target_is_directory=True)
envdir=pathlib.Path("/tmp/metric-env")
subprocess.run([sys.executable,"-m","venv","--without-pip","--system-site-packages",str(envdir)],check=True)
python=str(envdir/"bin/python")
pins=["numpy==2.5.4","scipy==1.18.1","scikit-learn==1.9.1","pyriemann==0.12","pandas==3.1.0","threadpoolctl==3.7.0"]
subprocess.run([sys.executable,"-m","pip","--python",python,"install","--quiet",*pins],check=True)
out=pathlib.Path("/kaggle/working/metric-results");out.mkdir(exist_ok=True)
env=dict(os.environ,METRIC_OUTPUT=str(out))
(out/"bundle-provenance.json").write_text(json.dumps(dict(metric_bundle_sha256="8c785a6e410628ecbf71b71558297ae6c25b4fadbd81db79933cb6cd0262d0b2",original_source_manifest_sha256=hashlib.sha256(raw).hexdigest(),input_manifest=input_manifest,source_manifest=manifest),indent=2))
print("Verified public data and author ShallowNet. All10people,nested early stopping,75min budget.",flush=True)

print("CUDA DEVICES",torch.cuda.device_count(),[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],flush=True)


## Data/split/loss/checkpoint and CUDA smoke tests

In [ ]:
subprocess.run([python,"-m","pytest","-q","test_core.py","test_training.py","--tb=short"],cwd=experiment,env=env,check=True)
smoke="""import sys;sys.path.insert(0,'src')
import torch
from neural_training import aligned_pair,fit,CONFIGS
from bci.data import synthetic
torch.set_num_threads(2)
w=synthetic(n_subjects=3,multi_session=0,runs=2,per_run=6)
x,_,_=aligned_pair(w,w)
model,r=fit(torch.as_tensor(x,device='cuda:0'),w,CONFIGS['shallow16_subject'],0,'/tmp/metric-cuda-smoke','cuda:0',epochs=1)
assert r['parameters']>0
print('CUDA weighted training and checkpoint smoke passed',flush=True)
"""
subprocess.run([python,"-c",smoke],cwd=experiment,env=env,check=True)


## Nested epoch selection and outer evaluation

In [ ]:
from IPython.display import display,Image
figures={}
process=subprocess.Popen([python,"-u",str(experiment/"run_gpu.py"),"--minutes","75"],cwd=experiment,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in process.stdout:
    print(line,end="",flush=True)
code=process.wait()
if code:raise RuntimeError(f"Metric research exited{code};checkpoints and partial predictions are saved")


## Completed-fit summary

In [ ]:
print((out/"REPORT.md").read_text())
print("Saved learning curves:",len(list(out.rglob("curves.png"))))